# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Macroeconomic Capture & Fiscal Crowding-Out

---
*Note: This notebook empirically quantifies fiscal crowding-out and the institutional friction of public spending. It transforms the Fundamental Government Budget Constraint Equation ($G_t + r B_{t-1} = T_t + B_t + \Delta M_t$) into algorithmic axioms. The computational infrastructure isolates the differential between state credit expansion and gross fixed capital formation in the private sector.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("flare")

In [ ]:
# Synthetic data loading
df = pd.read_csv('../data/macroeconomic_budget_synthetic.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)
df.head()

### 1. Modeling the Government Budget Constraint
We isolate the primary deficit and the debt service cost ($r B_{t-1}$) to understand the trajectory of government issuances.

In [ ]:
df['Primary_Deficit'] = df['Government_Spending'] - df['Tax_Revenue']
df['Debt_Service_Cost'] = df['Sovereign_Debt_Outstanding'].shift(1) * (df['Sovereign_Yield_10Y'] / 4) # Quarterly annualized
df['Total_Deficit'] = df['Primary_Deficit'] + df['Debt_Service_Cost']
df['Debt_Service_Cost'].fillna(0, inplace=True)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['Sovereign_Debt_Outstanding'], color='red', linewidth=2.5, label='Sovereign Debt ($B_t$)')
ax.bar(df.index, df['Primary_Deficit'], color='orange', alpha=0.5, width=60, label='Primary Deficit ($G_t - T_t$)')

ax.set_title("Structural Evolution of Sovereign Indebtedness", fontsize=14)
ax.set_ylabel("Monetary Volume")
ax.legend(loc='upper left')
plt.grid(True, alpha=0.2)
plt.show()

### 2. Topological Analysis of Crowding-Out (Private Sector)
We correlate the increase in *Sovereign Yield* brought about by the government's need for financing and its direct frictional impact on the private sector's Gross Capital Formation.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

# Private Sector Capital
ax1.plot(df.index, df['Private_Capital_Formation'], color='blue', linewidth=2, label='Private Capital Formation')

# Sovereign Yield indicating crowding out effect
ax2.plot(df.index, df['Sovereign_Yield_10Y'] * 100, color='yellow', linewidth=2, linestyle='--', label='10Y Sovereign Yield (%)')

ax1.set_title("Frictional Effect: Crowding-out on Capital Formation", fontsize=14)
ax1.set_ylabel("Private Capital ($)", color='blue')
ax2.set_ylabel("Sovereign Yield (%)", color='yellow')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper center')

plt.grid(True, alpha=0.2)
plt.show()

corr, pval = pearsonr(df['Sovereign_Yield_10Y'], df['Private_Capital_Formation'])
print(f"Yield-Private Correlation (Pearson): {corr:.4f} (p-value: {pval:.4e})")
print("Interpretation: A strong negative correlation indicates the algorithmic hijacking of credit by the State.")